In [1]:
print("welcome to ML for distributed_data frame")

welcome to ML for distributed_data frame


In [2]:
import sys
print(sys.version)

3.10.20 (main, Mar  3 2026, 09:24:47) [GCC 13.3.0]


In [3]:
try:
    spark.stop()
except:
    pass

In [4]:
from pyspark.sql import SparkSession
spark= SparkSession.builder\
       .appName("Distributed_ML")\
       .getOrCreate()
spark
       
  


In [5]:
spark.sparkContext.getConf().getAll()

[('spark.app.startTime', '1776986089807'),
 ('spark.executor.id', 'driver'),
 ('spark.driver.host', '10.0.2.15'),
 ('spark.driver.extraJavaOptions',
  '-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED --add-opens=java.security.jgss/sun.security.krb5=ALL-UNNAMED -Djdk.reflect.useDirectMethodHandle=false'),
 ('spark.app.submitTime', '1776986071158

In [6]:
spark.conf.set("spark.sql.shuffle.partitions", "50")

In [7]:
distributed_df=spark.read.parquet("/home/ubuntu/thesis/data/processed_data/system_state/distributed_data")
distributed_df.show(10)
distributed_df.count()

+--------------------+----------+-----+----------+----------+-----+-----+-----+---------+-------+--------------------+-----------+
|                Page|      Date|value|Moving_avg|Moving_std|lag_1|lag_2|trend|deviation|z_score|          percentile|state_label|
+--------------------+----------+-----+----------+----------+-----+-----+-----+---------+-------+--------------------+-----------+
|10._August_de.wik...|2015-07-01|   33|      33.0|      NULL| NULL| NULL| NULL|      0.0|   NULL|                 0.0|     Normal|
|10._August_de.wik...|2016-03-10|   28|      28.0|       0.0|   28|   28|  0.0|      0.0|   NULL|                 0.0|     Normal|
|10._August_de.wik...|2016-07-22|   34|      34.0|       0.0|   34|   34|  0.0|      0.0|   NULL|                 0.0|     Normal|
|10._August_de.wik...|2016-12-05|   24|     24.67|      0.58|   25|   25| -1.0|    -0.67|  -1.16|0.005565862708719...|     Normal|
|10._August_de.wik...|2016-12-24|   15|      17.0|      1.73|   18|   18| -3.0|    

57168842

In [8]:
from pyspark.sql.functions import col
print("Normal pages distributed dataset are " ,distributed_df.filter(col("state_label") == "Normal").count())
print("degraded pages in distributed dataset are ",distributed_df.filter(col("state_label") == "Degraded").count())
print("failure pages in distributed dataset are " ,distributed_df.filter(col("state_label") == "Failure").count())

Normal pages distributed dataset are  23017664


degraded pages in distributed dataset are  20020173


[Stage 11:===================================================>     (9 + 1) / 10]

failure pages in distributed dataset are  14131005


In [9]:
# ML-1 Base line model ( Logistic regression)

In [10]:
feature_columns=['value','Moving_avg','Moving_std','lag_1','lag_2','trend']
label='state_label'

In [11]:
from pyspark.ml.feature import StringIndexer
indexer=StringIndexer(inputCol='state_label',outputCol='label')
D_df=indexer.fit(distributed_df).transform(distributed_df)

D_df.show(5)

+--------------------+----------+-----+----------+----------+-----+-----+-----+---------+-------+--------------------+-----------+-----+
|                Page|      Date|value|Moving_avg|Moving_std|lag_1|lag_2|trend|deviation|z_score|          percentile|state_label|label|
+--------------------+----------+-----+----------+----------+-----+-----+-----+---------+-------+--------------------+-----------+-----+
|10._August_de.wik...|2015-07-01|   33|      33.0|      NULL| NULL| NULL| NULL|      0.0|   NULL|                 0.0|     Normal|  0.0|
|10._August_de.wik...|2016-03-10|   28|      28.0|       0.0|   28|   28|  0.0|      0.0|   NULL|                 0.0|     Normal|  0.0|
|10._August_de.wik...|2016-07-22|   34|      34.0|       0.0|   34|   34|  0.0|      0.0|   NULL|                 0.0|     Normal|  0.0|
|10._August_de.wik...|2016-12-05|   24|     24.67|      0.58|   25|   25| -1.0|    -0.67|  -1.16|0.005565862708719...|     Normal|  0.0|
|10._August_de.wik...|2016-12-24|   15|  

In [12]:
D_df.groupBy("label").count().show()

[Stage 18:===================================================>     (9 + 1) / 10]

+-----+--------+
|label|   count|
+-----+--------+
|  0.0|23017664|
|  1.0|20020173|
|  2.0|14131005|
+-----+--------+



In [13]:
D_df.printSchema()

root
 |-- Page: string (nullable = true)
 |-- Date: date (nullable = true)
 |-- value: string (nullable = true)
 |-- Moving_avg: double (nullable = true)
 |-- Moving_std: double (nullable = true)
 |-- lag_1: string (nullable = true)
 |-- lag_2: string (nullable = true)
 |-- trend: double (nullable = true)
 |-- deviation: double (nullable = true)
 |-- z_score: double (nullable = true)
 |-- percentile: double (nullable = true)
 |-- state_label: string (nullable = true)
 |-- label: double (nullable = false)



In [14]:
from pyspark.sql.functions import col
feature_columns=['value','Moving_avg','Moving_std','lag_1','lag_2','trend']
for col_name in feature_columns:
     D_df = D_df.withColumn(col_name, col(col_name).cast("double"))


D_df.show(5)

+--------------------+----------+-----+----------+----------+-----+-----+-----+---------+-------+--------------------+-----------+-----+
|                Page|      Date|value|Moving_avg|Moving_std|lag_1|lag_2|trend|deviation|z_score|          percentile|state_label|label|
+--------------------+----------+-----+----------+----------+-----+-----+-----+---------+-------+--------------------+-----------+-----+
|10._August_de.wik...|2015-07-01| 33.0|      33.0|      NULL| NULL| NULL| NULL|      0.0|   NULL|                 0.0|     Normal|  0.0|
|10._August_de.wik...|2016-03-10| 28.0|      28.0|       0.0| 28.0| 28.0|  0.0|      0.0|   NULL|                 0.0|     Normal|  0.0|
|10._August_de.wik...|2016-07-22| 34.0|      34.0|       0.0| 34.0| 34.0|  0.0|      0.0|   NULL|                 0.0|     Normal|  0.0|
|10._August_de.wik...|2016-12-05| 24.0|     24.67|      0.58| 25.0| 25.0| -1.0|    -0.67|  -1.16|0.005565862708719...|     Normal|  0.0|
|10._August_de.wik...|2016-12-24| 15.0|  

In [15]:
D_df.printSchema()

root
 |-- Page: string (nullable = true)
 |-- Date: date (nullable = true)
 |-- value: double (nullable = true)
 |-- Moving_avg: double (nullable = true)
 |-- Moving_std: double (nullable = true)
 |-- lag_1: double (nullable = true)
 |-- lag_2: double (nullable = true)
 |-- trend: double (nullable = true)
 |-- deviation: double (nullable = true)
 |-- z_score: double (nullable = true)
 |-- percentile: double (nullable = true)
 |-- state_label: string (nullable = true)
 |-- label: double (nullable = false)



In [16]:
from pyspark.sql.functions import col, isnan, when, count

D_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in D_df.columns
]).show()


[Stage 22:===================================================>     (9 + 1) / 10]

+----+----+-----+----------+----------+-----+------+-----+---------+-------+----------+-----------+-----+
|Page|Date|value|Moving_avg|Moving_std|lag_1| lag_2|trend|deviation|z_score|percentile|state_label|label|
+----+----+-----+----------+----------+-----+------+-----+---------+-------+----------+-----------+-----+
|   0|   0|    0|         0|     92168|92168|183964|92168|        0| 114257|         0|          0|    0|
+----+----+-----+----------+----------+-----+------+-----+---------+-------+----------+-----------+-----+



In [17]:
from pyspark.sql.functions import col
feature_columns=['value','Moving_avg','Moving_std','lag_1','lag_2','trend']
D_df_clean=D_df.dropna(subset=feature_columns)

In [18]:
from pyspark.sql.functions import col, isnan, when, count
D_df_clean.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in D_df_clean.columns
]).show()

[Stage 25:===================================================>     (9 + 1) / 10]

+----+----+-----+----------+----------+-----+-----+-----+---------+-------+----------+-----------+-----+
|Page|Date|value|Moving_avg|Moving_std|lag_1|lag_2|trend|deviation|z_score|percentile|state_label|label|
+----+----+-----+----------+----------+-----+-----+-----+---------+-------+----------+-----------+-----+
|   0|   0|    0|         0|         0|    0|    0|    0|        0|  20890|         0|          0|    0|
+----+----+-----+----------+----------+-----+-----+-----+---------+-------+----------+-----------+-----+



In [19]:
D_df_clean=D_df_clean.drop("z_score")
D_df_clean.show()
D_df_clean.count()

+--------------------+----------+-----+----------+----------+-----+-----+-----+---------+--------------------+-----------+-----+
|                Page|      Date|value|Moving_avg|Moving_std|lag_1|lag_2|trend|deviation|          percentile|state_label|label|
+--------------------+----------+-----+----------+----------+-----+-----+-----+---------+--------------------+-----------+-----+
|10._August_de.wik...|2016-03-10| 28.0|      28.0|       0.0| 28.0| 28.0|  0.0|      0.0|                 0.0|     Normal|  0.0|
|10._August_de.wik...|2016-07-22| 34.0|      34.0|       0.0| 34.0| 34.0|  0.0|      0.0|                 0.0|     Normal|  0.0|
|10._August_de.wik...|2016-12-05| 24.0|     24.67|      0.58| 25.0| 25.0| -1.0|    -0.67|0.005565862708719...|     Normal|  0.0|
|10._August_de.wik...|2016-12-24| 15.0|      17.0|      1.73| 18.0| 18.0| -3.0|     -2.0|0.005565862708719...|     Normal|  0.0|
|10._August_de.wik...|2015-07-21| 37.0|      60.0|     19.92| 71.0| 72.0|-34.0|    -23.0| 0.00927

56984878

In [20]:

from pyspark.sql.functions import min, max

D_df_clean.select(min("Date"), max("Date")).show()

[Stage 32:===================================================>     (9 + 1) / 10]

+----------+----------+
| min(Date)| max(Date)|
+----------+----------+
|2015-07-03|2016-12-31|
+----------+----------+



In [21]:
from pyspark.ml.feature import VectorAssembler
feature_columns=['value','Moving_avg','Moving_std','lag_1','lag_2','trend']
assembler=VectorAssembler(
    inputCols=feature_columns,
    outputCol="features")


D_df_model=assembler.transform(D_df_clean)
#D_df_model.count()

train = D_df_model.filter(col("Date") < "2016-06-01")
test  = D_df_model.filter(col("Date") >= "2016-06-01")

#print("Test:", test.count())

In [22]:
print("Train:", train.count())
print("Test:" , test.count())

Train: 33448317


[Stage 38:===================================================>     (9 + 1) / 10]

Test: 23536561


In [23]:
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withStd=True,
    withMean=True
)

scaler_model = scaler.fit(train)  
train = scaler_model.transform(train)
test = scaler_model.transform(test)

In [ ]:
#Logistic Regression
from pyspark.ml.classification import LogisticRegression
lr=LogisticRegression(
    featuresCol="scaled_features",
    labelCol="label",
    maxIter=50,
    regParam=0.01 
)
train=train.repartition(6)
train_lr = train.sample(0.6, seed=42) 
model = lr.fit(train_lr)

In [25]:
#  predictions = model.transform(test)
# #predictions.select("features", "label", "prediction").show()

In [ ]:
#ML metrics 

from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = evaluator.evaluate(predictions)
print("Accuracy for LR:", accuracy)

# evaluation for Recall
#from pyspark.ml.evaluation import MulticlassClassificationEvaluator
evaluator= MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
)
Recall=evaluator.evaluate(predictions)
print("Recall",Recall)

#from pyspark.ml.evaluation import MulticlassClassificationEvaluator
evaluator= MulticlassClassificationEvaluator(
             labelCol="label",
             predictionCol="prediction",
            metricName="weightedPrecision"
)

precesion=evaluator.evaluate(predictions)
print("precesion for LR:", precesion)

#from pyspark.ml.evaluation import MulticlassClassificationEvaluator
evaluator= MulticlassClassificationEvaluator(
            labelCol="label",
             predictionCol="prediction",
             metricName="f1"
)
f1= evaluator.evaluate(predictions)
print("F1 score for LR: ",f1)

In [27]:
# train.select("label").limit(10000).groupBy("label").count().show()

In [ ]:
#Random Forest 

from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

rf = RandomForestClassifier(
    featuresCol="scaled_features",
    labelCol="label",
    numTrees=50,
    maxDepth=5
)

from pyspark.ml.evaluation import MulticlassClassificationEvaluator
model = rf.fit(train)
predictions = model.transform(test)




[Stage 45:=================>                                       (3 + 3) / 10]

In [ ]:
#ML metrics for RF

#Accuracy
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)
accuracy=evaluator.evaluate(predictions)
print("Accuracy of RF:",accuracy)

# Precesion

evaluator= MulticlassClassificationEvaluator(
             labelCol="label",
             predictionCol="prediction",
            metricName="weightedPrecision"
)

precesion=evaluator.evaluate(predictions)
print("precesion of RF:", precesion)

# Recall
evaluator= MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
)
Recall=evaluator.evaluate(predictions)
print("Recall of RF:",Recall)

# F1 Score
evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)
f1=evaluator_f1.evaluate(predictions)
print("F1 Score of RF:", f1)


In [ ]:
# Decision tree
from pyspark.ml.classification import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    featuresCol="scaled_features",  
    labelCol="label",                
    maxDepth=5                      
)

model = dt.fit(train)
predictions = model.transform(test)

In [ ]:
from pyspark.ml.evaluation import  MulticlassClassificationEvaluator
#Accuracy
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)
accuracy=evaluator.evaluate(predictions)
print("Accuracy of Decision Tree:",accuracy)

# Precesion

evaluator= MulticlassClassificationEvaluator(
             labelCol="label",
             predictionCol="prediction",
            metricName="weightedPrecision"
)

precesion=evaluator.evaluate(predictions)
print("precesion of Decision Tree:", precesion)

# Recall
evaluator= MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
)
Recall=evaluator.evaluate(predictions)
print("Recall of Decision Tree:",Recall)

# F1 Score
evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)
f1=evaluator_f1.evaluate(predictions)
print("F1 Score of Decision Tree:", f1)

In [ ]:
# scalability calculation for distributed data set
df_1M = D_df_model.limit(1000000)
df_3M = D_df_model.limit(3000000)
df_5M = D_df_model  


df_1M_scaled = scaler_model.transform(df_1M)
df_3M_scaled = scaler_model.transform(df_3M)
df_5M_scaled = scaler_model.transform(df_5M)


print(df_1M.count())
print(df_3M.count())
print(df_5M.count())

In [ ]:
import time

def measure_training_time(df, model):
    start = time.time()
    model.fit(df)
    end = time.time()
    return end - start

In [ ]:
df_1M.printSchema()  

In [ ]:
# time_1M  = measure_training_time(df_1M_scaled, rf)
# time_3M = measure_training_time(df_3M_scaled, rf)
# time_5M   = measure_training_time(df_5M_scaled, rf)

# print("Small:",  time_1M)
# print("Medium:", time_3M)
# print("Full:", time_5M)

In [ ]:
# so for these data 
#  for 1M data it took nearly 242 seconds /1M
# for 3M dara it took nearly 623.83/3=207.94 sec/3M 
# for 5M data it took nearly 2119.433/5=423.8 sec/5M

# As the data increases then this distributed data set scales accordingly